# 🌾 Minería final Gravetal — 4 frentes (CRISP-DM)
**Segmentos + Fraude + Regímenes + Asignación.** Este notebook orquesta los scripts probados de la rama `mineria` y deja gráficos + conclusiones listos para defensa. Requiere `SNOWFLAKE_PASSWORD` (ver `requirements.txt`).

| Criterio | Dónde se cumple |
|---|---|
| 1 Dataset | features CSV + SQL en `sql/` |
| 2-3 Train/test | split-half (ARL/ARI), inyección sintética, baseline vs óptimo |
| 4 Métricas | silueta/DB/ARI, F1/AUC (ML), recall@k, MAE/R² |
| 5 Modelo | k-means, DBSCAN, IsolationForest, Benford, PuLP |
| 6 Negocio | precio diferencial, auditoría, mantenimiento, cobertura, asignación |
| 7-8 Resultados y visualización | perfiles, tops, este notebook |
| 9 Validación | estabilidad, sensibilidad, percentiles |
| 10 Nube | Snowflake en vivo + artefactos versionados |
| 11 Variables | diccionarios en cada script |

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
print('pwd Snowflake:', 'OK' if os.getenv('SNOWFLAKE_PASSWORD') else 'FALTA (solo CSV)')

## Frente 1 · Segmentos de productores (k-means + DBSCAN)

In [ ]:
%run mineria_productores.py

In [ ]:
seg = pd.read_csv('data/productores_segmentos_v2.csv')
feat = pd.read_csv('data/productores_features_v2.csv'); feat.columns = feat.columns.str.lower()
print('Segmentos:', seg['segmento'].value_counts().sort_index().to_dict(), '| atipicos:', int(seg['atipico_dbscan'].sum()))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
seg['segmento'].value_counts().sort_index().plot.bar(ax=ax[0]); ax[0].set_title('Tamano por segmento')
m = feat.merge(seg[['id_productor','segmento']], on='id_productor')
m.groupby('segmento')[['volumen_total_tn','max_merma_evento','pct_hum_crit']].mean().plot.bar(ax=ax[1]); ax[1].set_title('Perfil medio')
plt.tight_layout(); plt.show()

## Frente 2 · Fraude en báscula (score 0-100 + tipologías)

In [ ]:
%run mineria_fraude_bascula.py

In [ ]:
top = pd.read_csv('data/alerta_top.csv'); rs = pd.read_csv('data/riesgo_silo.csv')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
top['tipologia'].value_counts().plot.bar(ax=ax[0], color='#b91c1c'); ax[0].set_title('Top-5000 por tipologia')
top['severidad'].value_counts().reindex(['bajo','medio','alto','critico']).plot.bar(ax=ax[1], color='#ea580c'); ax[1].set_title('Por severidad')
rs.set_index(rs.columns[0])['mean'].head(10).plot.barh(ax=ax[2]); ax[2].set_title('Top silos (score medio)')
plt.tight_layout(); plt.show()

## Frente 3 · Regímenes IoT + CBOT

In [ ]:
%run mineria_regimenes.py

In [ ]:
ri = pd.read_csv('data/iot_sensor_regimen.csv'); rm = pd.read_csv('data/mercado_regimen_hora.csv')
print('Sensores por regimen dominante:'); print(ri.iloc[:, 1].value_counts().to_string() if ri.shape[1] > 1 else ri.head().to_string())
print(rm.groupby('regimen')[['volatilidad','volumen']].mean().round(4).to_string())

## Frente 4 · Asignación óptima a silos (PuLP)

In [ ]:
%run mineria_asignacion_silos.py

## Cierre · Conclusiones y despliegue
**Productores:** 2 segmentos (base estable vs intensivos con picos) + cuarentena 525. **Fraude:** score + 5 tipologías, severidad por percentil, top-k auditable. **Regímenes:** IoT K=5 operativos, CBOT K=2 (calmo/volátil). **Asignación:** ~9.9% menos merma (22.9 TN/semana).

**Despliegue:** `ORO.MINERIA_SEGMENTO_PRODUCTOR`, `ORO.MINERIA_ALERTA_BASCULA`, `ORO.MINERIA_REGIMEN_*` (batch diario/semanal) + re-ejecución por campaña. **Riesgos:** datos sintéticos (inventarios, fechas 1990/2099, contrato 1) → reglas DQ en Plata.